# Step 5E.2 — Feature attribution (gain/split importance), F0 vs F1

Phase 2 дизайну: native importance / CTI importance / Δimportance (redistribution).
Виконується ПІСЛЯ Step 5E.1 (overlap audit), який показав більшу нормалізовану залишкову
невизначеність CTI-service після врахування native-ознаки: 0.568 у CIC (H(cve_ctx_service|
protocol)/H(cve_ctx_service) = 1.2312/2.1665) проти 0.305 у UNSW (H(cve_ctx_service|service)/
H(cve_ctx_service) = 0.6542/2.1454) -- узгоджено з більшим ΔAP у CIC, але причинність ще не
доведена.

**Важлива методологічна примітка (не прихована, зазначена прямо):** Step 5D.2 ніколи не
зберігав файли моделей -- лише агреговані CSV метрик (перевірено у Step 5D.2b: `expected_
artifacts` включає лише `results/delta/topK` CSV, документовано "НЕ сирі test-прогнози чи
файли моделей"). Тому "feature importance без перенавчання" у буквальному сенсі
(завантажити збережений Booster) НЕМОЖЛИВЕ -- об'єкта моделі просто не існує на диску.

Натомість тут використовується ТОЙ САМИЙ frozen-протокол (явний порядок ознак з Step 5D.2e/f,
seed=42, ідентичні гіперпараметри), який щойно довів відтворення оригінальних AGGREGATE-метрик
(Δ~1e-16 для всіх 4 моделей, Step 5D.2b після виправлення). Це один-єдиний frozen прогін --
не hyperparameter search, не ретюнінг, не новий експеримент -- спосіб отримати Booster з тим
самим frozen протоколом і перевірити відтворення агрегованих frozen test-метрик (розділ 4
нижче), а не доказ побітової ідентичності внутрішньої структури дерев. Це прозоро
задокументовано тут, а не приховано за формулюванням "без перенавчання".

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path.home() / "notebooks" / "2 article" / "release" / "results"
ENRICHED_DIR = Path("/srv/ids_research/data/enriched")
AUDIT_OUT_DIR = RESULTS_DIR / "step5e2_feature_attribution"
AUDIT_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("RESULTS_DIR:", RESULTS_DIR, "exists:", RESULTS_DIR.exists())
print("ENRICHED_DIR:", ENRICHED_DIR, "exists:", ENRICHED_DIR.exists())


RESULTS_DIR: /home/ivan_r_opirskyy/notebooks/2 article/release/results exists: True
ENRICHED_DIR: /srv/ids_research/data/enriched exists: True


## 1. Frozen конфігурація і explicit ordered feature lists (буквально ті самі, що й Step 5D.2b після виправлення)

In [2]:
import lightgbm as lgb
from sklearn.metrics import average_precision_score

RANDOM_STATE = 42
LGB_PARAMS = dict(
    objective="binary", metric="None", seed=RANDOM_STATE,
    num_leaves=15, learning_rate=0.05, min_data_in_leaf=50,
)
NUM_BOOST_ROUND = 500
EARLY_STOPPING_ROUNDS = 30

def unweighted_average_precision(preds, train_data):
    y_true = train_data.get_label()
    return "ap", average_precision_score(y_true, preds), True

FROZEN_BOUNDARIES = {
    "CIC": dict(
        file="cic_ids2017_with_cve_context.parquet", time_col="Timestamp",
        label_col="attack_label", benign_value="BENIGN",
        boundary_1=pd.Timestamp("2017-07-06 09:28:00"),
        boundary_2=pd.Timestamp("2017-07-07 03:23:00"),
    ),
    "UNSW": dict(
        file="unsw_nb15_with_cve_context_dedup.parquet", time_col="Stime",
        label_col="attack_cat", benign_value="Normal",
        boundary_1=pd.Timestamp("2015-02-18 05:36:51"),
        boundary_2=pd.Timestamp("2015-02-18 08:58:36"),
    ),
}

FROZEN_F0_FEATURES = {
    "CIC": [
        "Flow Duration", "Total Fwd Packets", "Total Backward Packets", "Total Length of Fwd Packets",
        "Total Length of Bwd Packets", "Fwd Packet Length Max", "Fwd Packet Length Min",
        "Fwd Packet Length Mean", "Fwd Packet Length Std", "Bwd Packet Length Max",
        "Bwd Packet Length Min", "Bwd Packet Length Mean", "Bwd Packet Length Std",
        "Flow Bytes/s", "Flow Packets/s", "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max",
        "Flow IAT Min", "Fwd IAT Total", "Fwd IAT Mean", "Fwd IAT Std", "Fwd IAT Max",
        "Fwd IAT Min", "Bwd IAT Total", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Max",
        "Bwd IAT Min", "Fwd PSH Flags", "Bwd PSH Flags", "Fwd URG Flags", "Bwd URG Flags",
        "Fwd Header Length", "Bwd Header Length", "Fwd Packets/s", "Bwd Packets/s",
        "Min Packet Length", "Max Packet Length", "Packet Length Mean", "Packet Length Std",
        "Packet Length Variance", "FIN Flag Count", "SYN Flag Count", "RST Flag Count",
        "PSH Flag Count", "ACK Flag Count", "URG Flag Count", "CWE Flag Count", "ECE Flag Count",
        "Down/Up Ratio", "Average Packet Size", "Avg Fwd Segment Size", "Avg Bwd Segment Size",
        "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate", "Bwd Avg Bytes/Bulk",
        "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate", "Subflow Fwd Packets", "Subflow Fwd Bytes",
        "Subflow Bwd Packets", "Subflow Bwd Bytes", "Init_Win_bytes_forward",
        "Init_Win_bytes_backward", "act_data_pkt_fwd", "min_seg_size_forward", "Active Mean",
        "Active Std", "Active Max", "Active Min", "Idle Mean", "Idle Std", "Idle Max",
        "Idle Min", "protocol",
    ],
    "UNSW": [
        "state", "dur", "sbytes", "dbytes", "sttl", "dttl", "sloss", "dloss", "Sload", "Dload",
        "Spkts", "Dpkts", "swin", "dwin", "stcpb", "dtcpb", "smeansz", "dmeansz", "trans_depth",
        "res_bdy_len", "Sjit", "Djit", "Sintpkt", "Dintpkt", "tcprtt", "synack", "ackdat",
        "is_sm_ips_ports", "ct_state_ttl", "ct_flw_http_mthd", "is_ftp_login", "ct_ftp_cmd",
        "ct_srv_src", "ct_srv_dst", "ct_dst_ltm", "ct_src_ltm", "ct_src_dport_ltm",
        "ct_dst_sport_ltm", "ct_dst_src_ltm", "proto", "service",
    ],
}
F1_ADDITIONS = {
    "CIC": [
        "cve_ctx_service", "cve_ctx_pool_size", "cve_ctx_n_epss", "cve_ctx_epss_coverage",
        "cve_ctx_service_max_epss", "cve_ctx_n_generic_os_matches", "cve_ctx_n_specific_product_matches",
    ],
    "UNSW": [
        "cve_ctx_service", "cve_ctx_pool_size", "cve_ctx_epss_coverage",
        "cve_ctx_n_generic_os_matches", "cve_ctx_n_specific_product_matches",
    ],
}
FEATURE_SETS = {}
for dataset_key in ["CIC", "UNSW"]:
    FEATURE_SETS[(dataset_key, "F0")] = list(FROZEN_F0_FEATURES[dataset_key])
    FEATURE_SETS[(dataset_key, "F1")] = list(FROZEN_F0_FEATURES[dataset_key]) + list(F1_ADDITIONS[dataset_key])

EXPECTED_FEATURE_COUNTS = {("CIC", "F0"): 77, ("CIC", "F1"): 84, ("UNSW", "F0"): 41, ("UNSW", "F1"): 46}
for key, expected in EXPECTED_FEATURE_COUNTS.items():
    n = len(FEATURE_SETS[key])
    assert n == expected, f"{key}: n={n}, очікувалось {expected}"
print("✅ FEATURE_SETS (frozen explicit order) == очікувані 77/84/41/46")


✅ FEATURE_SETS (frozen explicit order) == очікувані 77/84/41/46


## 2. Свіже завантаження, frozen train/val/test split, categorical vocab fit-on-train

In [3]:
raw = {}
for dataset_key, cfg in FROZEN_BOUNDARIES.items():
    df = pd.read_parquet(ENRICHED_DIR / cfg["file"])
    raw[dataset_key] = df

splits = {}
for dataset_key, cfg in FROZEN_BOUNDARIES.items():
    df = raw[dataset_key].copy()
    df["_is_attack"] = (df[cfg["label_col"]] != cfg["benign_value"]).astype(int)

    if dataset_key == "UNSW" and "ct_ftp_cmd" in df.columns:
        df["ct_ftp_cmd"] = pd.to_numeric(df["ct_ftp_cmd"].replace(" ", np.nan), errors="coerce")

    t = pd.to_datetime(df[cfg["time_col"]], unit="s") if dataset_key == "UNSW" and pd.api.types.is_integer_dtype(df[cfg["time_col"]]) \
        else pd.to_datetime(df[cfg["time_col"]])

    train_mask = t < cfg["boundary_1"]
    val_mask = (t >= cfg["boundary_1"]) & (t < cfg["boundary_2"])
    test_mask = t >= cfg["boundary_2"]
    train_df, val_df, test_df = df[train_mask].copy(), df[val_mask].copy(), df[test_mask].copy()

    categorical_candidates = [c for c in FEATURE_SETS[(dataset_key, "F1")]
                              if not pd.api.types.is_numeric_dtype(df[c]) or df[c].dtype == bool]
    for col in categorical_candidates:
        train_categories = pd.Categorical(train_df[col]).categories
        train_df[col] = pd.Categorical(train_df[col], categories=train_categories)
        val_df[col] = pd.Categorical(val_df[col], categories=train_categories)
        test_df[col] = pd.Categorical(test_df[col], categories=train_categories)

    splits[dataset_key] = dict(train=train_df, val=val_df, test=test_df, categorical_candidates=categorical_candidates)
    n_train, n_pos = len(train_df), int(train_df["_is_attack"].sum())
    print(f"{dataset_key}: train n={n_train:,}  positive={n_pos:,}  categorical={categorical_candidates}")


CIC: train n=1,979,346  positive=265,848  categorical=['protocol', 'cve_ctx_service']
UNSW: train n=1,441,568  positive=49,346  categorical=['state', 'proto', 'service', 'cve_ctx_service']


## 3. Один frozen прогін для кожної з 4 моделей -- зберігаємо Booster-об'єкт (не лише метрики)

In [4]:
def train_frozen(dataset_key, feature_set_name):
    features = FEATURE_SETS[(dataset_key, feature_set_name)]
    train_df, val_df, test_df = splits[dataset_key]["train"], splits[dataset_key]["val"], splits[dataset_key]["test"]

    X_train, y_train = train_df[features], train_df["_is_attack"]
    X_val, y_val = val_df[features], val_df["_is_attack"]

    assert list(X_train.columns) == features, f"{dataset_key}-{feature_set_name}: X_train порядок розійшовся з frozen features!"
    assert list(X_val.columns) == features, f"{dataset_key}-{feature_set_name}: X_val порядок розійшовся з frozen features!"

    cat_cols = [c for c in features if X_train[c].dtype.name == "category"]
    n_pos_train = int(y_train.sum())
    n_neg_train = len(y_train) - n_pos_train
    w = n_neg_train / n_pos_train
    params = dict(LGB_PARAMS, scale_pos_weight=w)

    lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols if cat_cols else "auto")
    lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train, categorical_feature=cat_cols if cat_cols else "auto")

    model = lgb.train(
        params, lgb_train, num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[lgb_val], valid_names=["val"],
        feval=unweighted_average_precision,
        callbacks=[lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
                  lgb.log_evaluation(period=0)],
    )
    return model, features


models = {}
for dataset_key in ["CIC", "UNSW"]:
    for feature_set_name in ["F0", "F1"]:
        model, features = train_frozen(dataset_key, feature_set_name)
        models[(dataset_key, feature_set_name)] = dict(model=model, features=features)
        print(f"{dataset_key}-{feature_set_name}: best_iteration={model.best_iteration}  n_features={len(features)}")


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 265848, number of negative: 1713498
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.263011 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 14089
[LightGBM] [Info] Number of data points in the train set: 1979346, number of used features: 69
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.134311 -> initscore=-1.863367
[LightGBM] [Info] Start training from score -1.863367
CIC-F0: best_iteration=4  n_features=77
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 265848, number of negative: 1713498
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing 

## 4. Reproducibility audit ПЕРЕД importance -- best_iteration НЕДОСТАТНІЙ доказ

Критичне уточнення: однаковий `best_iteration` НЕ гарантує ідентичності Booster -- ми вже
мали прецедент (UNSW, Step 5D.2b), де той самий `best_iteration`/`Total Bins` міг виникнути
при різних деревах через порядок колонок. Тому тут перевіряється мінімум: `best_iteration`,
`test_ap`, `test_roc_auc` -- проти frozen референсних значень Step 5D.2 (розділ "Фінальні
frozen результати Step 5D.2" з попереднього аудиту). `val_ap`/`val_roc_auc` обчислюються і
виводяться як додатковий діагностичний сигнал, БЕЗ порівняння з референсом -- точні
збережені validation-значення оригінального Step 5D.2 прогону не зафіксовані окремо в
жодному з попередніх аудитів цієї розмови, тому вигадувати їх як "frozen reference" тут
НЕ будемо.

Якщо цей блок НЕ проходить -- importance-аналіз нижче не має сенсу (модель не та, що дала
frozen результати), і Phase 2 зупиняється тут.

In [5]:
from sklearn.metrics import roc_auc_score

FROZEN_REFERENCE = {
    ("CIC", "F0"): dict(best_iteration=4, test_ap=0.710369, test_roc_auc=0.740553),
    ("CIC", "F1"): dict(best_iteration=118, test_ap=0.974028, test_roc_auc=0.982162),
    ("UNSW", "F0"): dict(best_iteration=33, test_ap=0.973652, test_roc_auc=0.997671),
    ("UNSW", "F1"): dict(best_iteration=290, test_ap=0.983969, test_roc_auc=0.998247),
}

repro_rows = []
for (dataset_key, feature_set_name), payload in models.items():
    model, features = payload["model"], payload["features"]
    val_df, test_df = splits[dataset_key]["val"], splits[dataset_key]["test"]
    y_val, y_test = val_df["_is_attack"].to_numpy(), test_df["_is_attack"].to_numpy()

    p_val = model.predict(val_df[features], num_iteration=model.best_iteration)
    p_test = model.predict(test_df[features], num_iteration=model.best_iteration)

    val_ap = average_precision_score(y_val, p_val)
    val_roc = roc_auc_score(y_val, p_val)
    test_ap = average_precision_score(y_test, p_test)
    test_roc = roc_auc_score(y_test, p_test)

    ref = FROZEN_REFERENCE[(dataset_key, feature_set_name)]
    repro_rows.append(dict(
        dataset=dataset_key, feature_set=feature_set_name,
        best_iteration=int(model.best_iteration), ref_best_iteration=ref["best_iteration"],
        val_ap=val_ap, val_roc_auc=val_roc,  # немає frozen референсу -- лише діагностика
        test_ap=test_ap, ref_test_ap=ref["test_ap"],
        test_roc_auc=test_roc, ref_test_roc_auc=ref["test_roc_auc"],
        best_iteration_match=int(model.best_iteration) == ref["best_iteration"],
        test_ap_match=bool(np.isclose(test_ap, ref["test_ap"], rtol=0, atol=1e-6)),
        test_roc_match=bool(np.isclose(test_roc, ref["test_roc_auc"], rtol=0, atol=1e-6)),
    ))

repro_df = pd.DataFrame(repro_rows)
print(repro_df.to_string(index=False))

assert repro_df["best_iteration_match"].all(), "best_iteration НЕ збігається з frozen референсом -- reproduction FAILED"
assert repro_df["test_ap_match"].all(), "test_ap НЕ збігається з frozen референсом -- reproduction FAILED"
assert repro_df["test_roc_match"].all(), "test_roc_auc НЕ збігається з frozen референсом -- reproduction FAILED"

repro_df.to_csv(AUDIT_OUT_DIR / "reproducibility_audit_before_importance.csv", index=False)
print("\n✅ Frozen Step 5D.2 reproduction підтверджено (best_iteration + test_ap + test_roc_auc) -- "
      "лише ПІСЛЯ цього importance-аналіз нижче має сенс.")


dataset feature_set  best_iteration  ref_best_iteration   val_ap  val_roc_auc  test_ap  ref_test_ap  test_roc_auc  ref_test_roc_auc  best_iteration_match  test_ap_match  test_roc_match
    CIC          F0               4                   4 0.778550     0.823532 0.710369     0.710369      0.740553          0.740553                  True           True            True
    CIC          F1             118                 118 0.676896     0.765135 0.974028     0.974028      0.982162          0.982162                  True           True            True
   UNSW          F0              33                  33 0.964633     0.997017 0.973652     0.973652      0.997671          0.997671                  True           True            True
   UNSW          F1             290                 290 0.979776     0.997882 0.983969     0.983969      0.998247          0.998247                  True           True            True

✅ Frozen Step 5D.2 reproduction підтверджено (best_iteration + test_ap + t

## 5. Feature importance (gain і split) для кожної з 4 моделей

In [6]:
importance_rows = []
for (dataset_key, feature_set_name), payload in models.items():
    model, features = payload["model"], payload["features"]
    gain = model.feature_importance(importance_type="gain")
    split = model.feature_importance(importance_type="split")
    total_gain = gain.sum()
    for feat, g, s in zip(features, gain, split):  # ORIGINAL frozen names/order, НЕ model.feature_name()
        # (LightGBM внутрішньо замінює пробіли в іменах ознак на "_", напр. "Flow Duration" ->
        # "Flow_Duration" -- model.feature_name() повернув би санітизовані імена, що зламало б
        # приєднання назад до FROZEN_F0_FEATURES у розділі 6. Масиви gain/split повертаються в
        # ТОМУ САМОМУ порядку, в якому features передавались у lgb.Dataset, тож zip(features, ...)
        # коректний незалежно від внутрішньої санітизації імен LightGBM.)
        importance_rows.append(dict(
            dataset=dataset_key, feature_set=feature_set_name, feature=feat,
            gain=float(g), split=int(s),
            gain_share=float(g / total_gain) if total_gain > 0 else 0.0,
            is_cti=feat.startswith("cve_ctx_"),
        ))

importance_df = pd.DataFrame(importance_rows)
print(f"Побудовано importance_df: {len(importance_df)} рядків (feature x model)")
importance_df.head()


Побудовано importance_df: 248 рядків (feature x model)


,dataset,feature_set,feature,gain,split,gain_share,is_cti
0,CIC,F0,Flow Duration,0.0,0,0.0,False
1,CIC,F0,Total Fwd Packets,0.0,0,0.0,False
2,CIC,F0,Total Backward Packets,0.0,0,0.0,False
3,CIC,F0,Total Length of Fwd Packets,0.0,0,0.0,False
4,CIC,F0,Total Length of Bwd Packets,0.0,0,0.0,False


## 6. Частка сукупного gain, що припадає на CTI (F1-моделі) -- і внесок cve_ctx_service окремо

In [7]:
print("Частка сукупного gain, що припадає на CTI (усі cve_ctx_*), F1-моделі:")
cti_share_rows = []
for dataset_key in ["CIC", "UNSW"]:
    sub = importance_df[(importance_df["dataset"] == dataset_key) & (importance_df["feature_set"] == "F1")]
    cti_gain_share = sub.loc[sub["is_cti"], "gain_share"].sum()
    cve_ctx_service_share = sub.loc[sub["feature"] == "cve_ctx_service", "gain_share"].sum()
    other_cti_share = cti_gain_share - cve_ctx_service_share
    cti_share_rows.append(dict(
        dataset=dataset_key, total_cti_gain_share=cti_gain_share,
        cve_ctx_service_gain_share=cve_ctx_service_share, other_cti_gain_share=other_cti_share,
    ))
    print(f"  {dataset_key}-F1: CTI total={cti_gain_share:.4f}  cve_ctx_service={cve_ctx_service_share:.4f}  "
          f"решта CTI={other_cti_share:.4f}")

cti_share_df = pd.DataFrame(cti_share_rows)
print()
print(cti_share_df.to_string(index=False))


Частка сукупного gain, що припадає на CTI (усі cve_ctx_*), F1-моделі:
  CIC-F1: CTI total=0.8198  cve_ctx_service=0.8194  решта CTI=0.0004
  UNSW-F1: CTI total=0.0049  cve_ctx_service=0.0043  решта CTI=0.0006

dataset  total_cti_gain_share  cve_ctx_service_gain_share  other_cti_gain_share
    CIC              0.819793                    0.819384              0.000409
   UNSW              0.004900                    0.004278              0.000622


## 7. Перерозподіл importance НАТИВНИХ ознак: F0 -> F1 (ΔI_j = I_j(F1) - I_j(F0))

Топ-10 native-ознак за абсолютною зміною gain_share після додавання CTI -- показує, чи CTI
"забирає" сигнал у нативних ознак (перерозподіл), чи додає окремий сигнал без витіснення
наявного.

In [8]:
redistribution_rows = []
for dataset_key in ["CIC", "UNSW"]:
    f0_sub = importance_df[(importance_df["dataset"] == dataset_key) & (importance_df["feature_set"] == "F0")]
    f1_sub = importance_df[(importance_df["dataset"] == dataset_key) & (importance_df["feature_set"] == "F1")]
    f0_map = dict(zip(f0_sub["feature"], f0_sub["gain_share"]))
    f1_map = dict(zip(f1_sub["feature"], f1_sub["gain_share"]))
    for feat in FROZEN_F0_FEATURES[dataset_key]:
        i_f0 = f0_map.get(feat, 0.0)
        i_f1 = f1_map.get(feat, 0.0)
        redistribution_rows.append(dict(dataset=dataset_key, feature=feat,
                                        gain_share_F0=i_f0, gain_share_F1=i_f1, delta=i_f1 - i_f0))

redistribution_df = pd.DataFrame(redistribution_rows)
for dataset_key in ["CIC", "UNSW"]:
    print(f"\n{dataset_key}: топ-10 native-ознак за |Δgain_share| (F1-F0):")
    sub = redistribution_df[redistribution_df["dataset"] == dataset_key].copy()
    sub["abs_delta"] = sub["delta"].abs()
    print(sub.sort_values("abs_delta", ascending=False).head(10)[["feature", "gain_share_F0", "gain_share_F1", "delta"]].to_string(index=False))

sum_delta = redistribution_df.groupby("dataset")["delta"].sum()
print("\nСума Δgain_share по всіх native-ознаках (має бути ~ -total_cti_gain_share, "
      "оскільки gain_share нормовано на 1.0 в межах кожної моделі):")
print(sum_delta)



CIC: топ-10 native-ознак за |Δgain_share| (F1-F0):
                feature  gain_share_F0  gain_share_F1     delta
      Min Packet Length       0.293892       0.014257 -0.279635
  Bwd Packet Length Std       0.183137       0.006295 -0.176842
Init_Win_bytes_backward       0.223810       0.072563 -0.151247
     Packet Length Mean       0.090214       0.000280 -0.089934
 Init_Win_bytes_forward       0.097628       0.018023 -0.079605
          Flow IAT Mean       0.027192       0.000742 -0.026450
           Flow IAT Min       0.027530       0.006517 -0.021013
  Fwd Packet Length Std       0.018464       0.006608 -0.011856
           Flow Bytes/s       0.008108       0.000531 -0.007577
           Bwd IAT Mean       0.007409       0.000601 -0.006808

UNSW: топ-10 native-ознак за |Δgain_share| (F1-F0):
       feature  gain_share_F0  gain_share_F1     delta
          sttl       0.988074       0.966122 -0.021952
  ct_state_ttl       0.001190       0.005748  0.004558
         proto       0.004

## 8. Збереження артефактів

In [9]:
importance_df.to_csv(AUDIT_OUT_DIR / "feature_importance_all_models.csv", index=False)
cti_share_df.to_csv(AUDIT_OUT_DIR / "cti_gain_share_summary.csv", index=False)
redistribution_df.to_csv(AUDIT_OUT_DIR / "native_feature_redistribution.csv", index=False)

best_iterations = {f"{k[0]}_{k[1]}": int(v["model"].best_iteration) for k, v in models.items()}
summary = dict(
    note="Одноразовий frozen прогін (не ретюнінг) -- Booster-об'єкти оригінально ніколи не збережено Step 5D.2",
    best_iterations=best_iterations,
    cti_gain_share=cti_share_df.to_dict(orient="records"),
)
with open(AUDIT_OUT_DIR / "step5e2_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Збережено в {AUDIT_OUT_DIR}:")
for p in sorted(AUDIT_OUT_DIR.glob("*")):
    print(f"  {p.name}")


Збережено в /home/ivan_r_opirskyy/notebooks/2 article/release/results/step5e2_feature_attribution:
  cti_gain_share_summary.csv
  feature_importance_all_models.csv
  native_feature_redistribution.csv
  reproducibility_audit_before_importance.csv
  step5e2_summary.json


## Підсумок Phase 2

Заповнюється фактичним виводом після запуску на сервері. Референсні `best_iteration` зі
Step 5D.2 (для звірки з розділом 4): CIC-F0=4, CIC-F1=118, UNSW-F0=33, UNSW-F1=290.

Важливе уточнення щодо термінології: розділ 4 підтверджує -- або спростовує -- що поточний
frozen rerun ВІДТВОРЮЄ результати Step 5D.2 (`best_iteration` + `test_ap` + `test_roc_auc`
разом, це reproducibility audit, а не вибір моделі). Це НЕ доказ, що Booster "ідентичний"
оригіналу в сенсі побітової рівності дерев -- такого доказу в нас немає і не потрібно; нам
достатньо, що метрики відтворюються з точністю, вказаною в audit-блоці (atol=1e-6), а
`val_ap`/`val_roc_auc` виводяться як додаткова діагностика без порівняння з референсом
(точні збережені validation-значення оригінального прогону Step 5D.2 в нас не зафіксовані).
Лише якщо цей audit проходить, має сенс інтерпретувати importance нижче як importance
ТІЄЇ Ж моделі, що дала frozen результати.

За умови, що audit пройдено: чи `cve_ctx_service` домінує серед CTI-ознак в обох датасетах,
чи внесок розподілений між кількома CTI-ознаками; чи перерозподіл нативних ознак (розділ 7)
показує витіснення сигналу в UNSW (де overlap високий, за Step 5E.1) сильніше, ніж у CIC.

Критичне методологічне обмеження: gain/split importance показує ЛИШЕ, де модель
використовує CTI-сигнал (кореляційна структура використання), а НЕ причинний внесок --
через корельовані/надлишкові ознаки висока gain-частка `cve_ctx_service` сама по собі НЕ
дозволяє висновок «`cve_ctx_service` має найбільший gain -> саме він пояснює ΔAP». Це
завелике твердження на основі самого лише feature attribution.

Те, чи Phase 3 (targeted ablation: F0 + cve_ctx_service окремо від F0 + решта CTI, окремо
від повного F1) потрібен для остаточного причинного розкладання внеску ΔAP, визначається
після перегляду фактичної картини Phase 2 -- Phase 3 не запускається спекулятивно
паралельно з Phase 2.